# Reshaping and Pivot tablespivot() and pivot_table()


pandas provides methods for manipulating a **`Series`** and **`DataFrame`** to alter the representation of the data for further data processing or data summarization.

* **`pivot()`** and **`pivot_table()`**: Group unique values within one or more discrete categories.
* **`stack()`** and **`unstack()`**: Pivot a column or row level to the opposite axis respectively.
* **`melt()`** and **`wide_to_long()`**: Unpivot a wide **`DataFrame`** to a long format.
* **`get_dummies()`** and **`from_dummies()`**: Conversions with indicator variables.
* **`explode()`**: Convert a column of list-like values to individual rows.
* **`crosstab()`**: Calculate a cross-tabulation of multiple 1 dimensional factor arrays.
* **`cut()`**: Transform continuous variables to discrete, categorical values
* **`factorize()`**: Encode 1 dimensional variables into integer labels.



### pivot() and pivot_table()
**pivot()**

Data is often stored in so-called “stacked” or “record” format. In a “record” or “wide” format, typically there is one row for each subject. In the “stacked” or “long” format there are multiple rows for each subject where applicable.

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = {
   "value": range(12),
   "variable": ["A"] * 3 + ["B"] * 3 + ["C"] * 3 + ["D"] * 3,
   "date": pd.to_datetime(["2020-01-03", "2020-01-04", "2020-01-05"] * 4)
}


df = pd.DataFrame(data)

To perform time series operations with each unique variable, a better representation would be where the `columns` are the unique variables and an `index` of dates identifies individual observations. To reshape the data into this form, we use the `DataFrame.pivot()` method (also implemented as a top level function `pivot()`):

In [3]:
pivoted = df.pivot(index="date", columns="variable", values="value")
pivoted

variable,A,B,C,D
date,,,,
2020-01-03,0,3,6,9
2020-01-04,1,4,7,10
2020-01-05,2,5,8,11


If the values argument is omitted, and the input DataFrame has more than one column of values which are not used as column or index inputs to pivot(), then the resulting “pivoted” DataFrame will have hierarchical columns whose topmost level indicates the respective value column:

In [4]:
df["value2"] = df["value"] * 2
pivoted = df.pivot(index="date", columns="variable")
pivoted

value           value2            
variable       A  B  C   D      A   B   C   D
date                                         
2020-01-03     0  3  6   9      0   6  12  18
2020-01-04     1  4  7  10      2   8  14  20
2020-01-05     2  5  8  11      4  10  16  22

In [5]:
# You can then select subsets from the pivoted DataFrame:
pivoted["value2"]

variable,A,B,C,D
date,,,,
2020-01-03,0,6,12,18
2020-01-04,2,8,14,20
2020-01-05,4,10,16,22


**pivot_table()**

While `pivot()` provides general purpose pivoting with various data types, pandas also provides `pivot_table()` or `pivot_table()` for pivoting with aggregation of numeric data.

The function `pivot_table()` can be used to create spreadsheet-style pivot tables. See the cookbook for some advanced strategies.

In [6]:
import datetime
df = pd.DataFrame(
    {
        "A": ["one", "one", "two", "three"] * 6,
        "B": ["A", "B", "C"] * 8,
        "C": ["foo", "foo", "foo", "bar", "bar", "bar"] * 4,
        "D": np.random.randn(24),
        "E": np.random.randn(24),
        "F": [datetime.datetime(2013, i, 1) for i in range(1, 13)]
        + [datetime.datetime(2013, i, 15) for i in range(1, 13)],
    }
)

df

,A,B,C,D,E,F
0,one,A,foo,-0.600849,-0.436166,2013-01-01
1,one,B,foo,-0.438649,-1.628142,2013-02-01
2,two,C,foo,-0.010020,-0.647889,2013-03-01
3,three,A,bar,-2.807567,2.275648,2013-04-01
4,one,B,bar,0.340051,-1.864623,2013-05-01
5,one,C,bar,-1.438701,0.104725,2013-06-01
6,two,A,foo,-1.606756,0.373609,2013-07-01
7,three,B,foo,1.016491,-0.879408,2013-08-01
8,one,C,foo,0.059850,-1.700648,2013-09-01
9,one,A,bar,-1.160883,-0.361994,2013-10-01


In [7]:
pd.pivot_table(df, values="D", index=["A", "B"], columns=["C"])

C             bar       foo
A     B                    
one   A -0.335915  0.032104
      B -0.576067 -0.150163
      C  0.205614  0.373626
three A -1.434189       NaN
      B       NaN  0.808890
      C -0.423432       NaN
two   A       NaN -0.357437
      B -0.690576       NaN
      C       NaN  0.490133

In [8]:
pd.pivot_table(
    df, values=["D", "E"],
    index=["B"],
    columns=["A", "C"],
    aggfunc="sum",
)

D                                                           E  \
A       one               three                 two                 one   
C       bar       foo       bar       foo       bar       foo       bar   
B                                                                         
A -0.671830  0.064208 -2.868379       NaN       NaN -0.714874 -0.168364   
B -1.152134 -0.300326       NaN  1.617779 -1.381153       NaN  0.583810   
C  0.411229  0.747253 -0.846864       NaN       NaN  0.980265 -0.657730   

                                                     
A               three                 two            
C       foo       bar       foo       bar       foo  
B                                                    
A -0.161648  2.366034       NaN       NaN -0.194928  
B -2.335995       NaN -1.310576 -0.973053       NaN  
C -0.746299  0.193933       NaN       NaN -2.023400

In [9]:
pd.pivot_table(
    df, values="E",
    index=["B", "C"],
    columns=["A"],
    aggfunc=["sum", "mean"],
)

sum                          mean                    
A           one     three       two       one     three       two
B C                                                              
A bar -0.168364  2.366034       NaN -0.084182  1.183017       NaN
  foo -0.161648       NaN -0.194928 -0.080824       NaN -0.097464
B bar  0.583810       NaN -0.973053  0.291905       NaN -0.486526
  foo -2.335995 -1.310576       NaN -1.167998 -0.655288       NaN
C bar -0.657730  0.193933       NaN -0.328865  0.096967       NaN
  foo -0.746299       NaN -2.023400 -0.373149       NaN -1.011700

The result is a DataFrame potentially having a MultiIndex on the index or column. If the values column name is not given, the pivot table will include all of the data in an additional level of hierarchy in the columns:

In [10]:
pd.pivot_table(df[["A", "B", "C", "D", "E"]], index=["A", "B"], columns=["C"])

D                   E          
C             bar       foo       bar       foo
A     B                                        
one   A -0.335915  0.032104 -0.084182 -0.080824
      B -0.576067 -0.150163  0.291905 -1.167998
      C  0.205614  0.373626 -0.328865 -0.373149
three A -1.434189       NaN  1.183017       NaN
      B       NaN  0.808890       NaN -0.655288
      C -0.423432       NaN  0.096967       NaN
two   A       NaN -0.357437       NaN -0.097464
      B -0.690576       NaN -0.486526       NaN
      C       NaN  0.490133       NaN -1.011700

In [11]:
# Also, you can use Grouper for index and columns keywords.
pd.pivot_table(df, values="D", index=pd.Grouper(freq="ME", key="F"), columns="C")

C,bar,foo
F,,
2013-01-31,NaN,0.032104
2013-02-28,NaN,-0.150163
2013-03-31,NaN,0.490133
2013-04-30,-1.434189,NaN
2013-05-31,-0.576067,NaN
2013-06-30,0.205614,NaN
2013-07-31,NaN,-0.357437
2013-08-31,NaN,0.808890
2013-09-30,NaN,0.373626


**Adding Margins**

Passing margins=True to pivot_table() will add a row and column with an All label with partial group aggregates across the categories on the rows and columns:

In [12]:
table = df.pivot_table(
    index=["A", "B"],
    columns="C",
    values=["D", "E"],
    margins=True,
    aggfunc="std"
)


table

D                             E                    
C             bar       foo       All       bar       foo       All
A     B                                                            
one   A  1.166681  0.895130  0.875184  0.392885  0.502530  0.368287
      B  1.295586  0.407981  0.821865  3.049791  0.650742  1.987964
      C  2.325413  0.443746  1.370241  0.613188  1.877366  1.140536
three A  1.942249       NaN  1.942249  1.545214       NaN  1.545214
      B       NaN  0.293593  0.293593       NaN  0.316954  0.316954
      C  0.488515       NaN  0.488515  0.051173       NaN  0.051173
two   A       NaN  1.766803  1.766803       NaN  0.666198  0.666198
      B  0.209648       NaN  0.209648  1.992758       NaN  1.992758
      C       NaN  0.707323  0.707323       NaN  0.514506  0.514506
All      1.182122  0.784273  1.051716  1.339419  0.805196  1.134667

In [13]:
# Additionally, you can call DataFrame.stack() to display a pivoted DataFrame as having a multi-level index:
table.stack()

D         E
A     B C                      
one   A bar  1.166681  0.392885
        foo  0.895130  0.502530
        All  0.875184  0.368287
      B bar  1.295586  3.049791
        foo  0.407981  0.650742
        All  0.821865  1.987964
      C bar  2.325413  0.613188
        foo  0.443746  1.877366
        All  1.370241  1.140536
three A bar  1.942249  1.545214
        foo       NaN       NaN
        All  1.942249  1.545214
      B bar       NaN       NaN
        foo  0.293593  0.316954
        All  0.293593  0.316954
      C bar  0.488515  0.051173
        foo       NaN       NaN
        All  0.488515  0.051173
two   A bar       NaN       NaN
        foo  1.766803  0.666198
        All  1.766803  0.666198
      B bar  0.209648  1.992758
        foo       NaN       NaN
        All  0.209648  1.992758
      C bar       NaN       NaN
        foo  0.707323  0.514506
        All  0.707323  0.514506
All     bar  1.182122  1.339419
        foo  0.784273  0.805196
        All  1.051716  1.134667

### stack() and unstack()
Closely related to the **`pivot()`** method are the related **`stack()`** and **`unstack()`** methods available on **`Series`** and **`DataFrame`**. These methods are designed to work together with **`MultiIndex`** objects (see the section on hierarchical indexing).

* **`stack()`**: “pivot” a level of the (possibly hierarchical) column labels, returning a **`DataFrame`** with an index with a new inner-most level of row labels.
* **`unstack()`**: (inverse operation of **`stack()`**) “pivot” a level of the (possibly hierarchical) row index to the column axis, producing a reshaped **`DataFrame`** with a new inner-most level of column labels.


In [14]:
tuples = [
   ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
   ["one", "two", "one", "two", "one", "two", "one", "two"],
]

index = pd.MultiIndex.from_arrays(tuples, names=["first", "second"])
df = pd.DataFrame(np.random.randn(8, 2), index=index, columns=["A", "B"])
df2 = df[:4]
df2

A         B
first second                    
bar   one     0.166801  0.564108
      two     1.283708 -1.188816
baz   one    -0.963693 -0.262022
      two     0.681834 -0.548250

The **`stack()`** function “compresses” a level in the **`DataFrame`** columns to produce either:

* A **`Series`**, in the case of an **`Index`** in the columns.
* A **`DataFrame`**, in the case of a **`MultiIndex`** in the columns.

If the columns have a **`MultiIndex`**, you can choose which level to stack. The stacked level becomes the new lowest level in a **`MultiIndex`** on the columns:


In [15]:
stacked = df2.stack()

stacked

first  second   
bar    one     A    0.166801
               B    0.564108
       two     A    1.283708
               B   -1.188816
baz    one     A   -0.963693
               B   -0.262022
       two     A    0.681834
               B   -0.548250
dtype: float64

In [16]:
# With a “stacked” DataFrame or Series (having a MultiIndex as the index), the inverse operation of stack() is unstack(), which by default unstacks the last level:
stacked.unstack()

A         B
first second                    
bar   one     0.166801  0.564108
      two     1.283708 -1.188816
baz   one    -0.963693 -0.262022
      two     0.681834 -0.548250

In [17]:
stacked.unstack(1)

second        one       two
first                      
bar   A  0.166801  1.283708
      B  0.564108 -1.188816
baz   A -0.963693  0.681834
      B -0.262022 -0.548250

In [18]:
stacked.unstack(0)

first          bar       baz
second                      
one    A  0.166801 -0.963693
       B  0.564108 -0.262022
two    A  1.283708  0.681834
       B -1.188816 -0.548250

In [19]:
# If the indexes have names, you can use the level names instead of specifying the level numbers:
stacked.unstack("second")

second        one       two
first                      
bar   A  0.166801  1.283708
      B  0.564108 -1.188816
baz   A -0.963693  0.681834
      B -0.262022 -0.548250

In [20]:
# Notice that the stack() and unstack() methods implicitly sort the index levels involved. 
# Hence a call to stack() and then unstack(), or vice versa, will result in a sorted copy of the original DataFrame or Series:

index = pd.MultiIndex.from_product([[2, 1], ["a", "b"]])
df = pd.DataFrame(np.random.randn(4), index=index, columns=["A"])
df

A
2 a  0.435650
  b  1.629457
1 a -0.094907
  b -2.492746

**Multiple levels**

You may also stack or unstack more than one level at a time by passing a list of levels, in which case the end result is as if each level in the list were processed individually.

In [21]:
columns = pd.MultiIndex.from_tuples(
    [
        ("A", "cat", "long"),
        ("B", "cat", "long"),
        ("A", "dog", "short"),
        ("B", "dog", "short"),
    ],
    names=["exp", "animal", "hair_length"],
)

df = pd.DataFrame(np.random.randn(4, 4), columns=columns)
df

exp,A,B,A,B
animal,cat,cat,dog,dog
hair_length,long,long,short,short
0,-1.424141,-1.286828,-3.042640,-0.105105
1,0.078058,0.951788,-0.803569,1.579763
2,0.156693,0.531594,0.234046,-1.084008
3,-0.620109,-1.014952,1.322144,-0.195691


In [22]:
df.stack(level=["animal", "hair_length"])

exp                          A         B
  animal hair_length                    
0 cat    long        -1.424141 -1.286828
  dog    short       -3.042640 -0.105105
1 cat    long         0.078058  0.951788
  dog    short       -0.803569  1.579763
2 cat    long         0.156693  0.531594
  dog    short        0.234046 -1.084008
3 cat    long        -0.620109 -1.014952
  dog    short        1.322144 -0.195691

In [23]:
# The list of levels can contain either level names or level numbers but not a mixture of the two.
# df.stack(level=['animal', 'hair_length'])
# from above is equivalent to:
df.stack(level=[1, 2])

exp                          A         B
  animal hair_length                    
0 cat    long        -1.424141 -1.286828
  dog    short       -3.042640 -0.105105
1 cat    long         0.078058  0.951788
  dog    short       -0.803569  1.579763
2 cat    long         0.156693  0.531594
  dog    short        0.234046 -1.084008
3 cat    long        -0.620109 -1.014952
  dog    short        1.322144 -0.195691

**Missing data**

Unstacking can result in missing values if subgroups do not have the same set of labels. By default, missing values will be replaced with the default fill value for that data type.

In [24]:
columns = pd.MultiIndex.from_tuples(
    [
        ("A", "cat"),
        ("B", "dog"),
        ("B", "cat"),
        ("A", "dog"),
    ],
    names=["exp", "animal"],
)


index = pd.MultiIndex.from_product(
    [("bar", "baz", "foo", "qux"), ("one", "two")], names=["first", "second"]
)


df = pd.DataFrame(np.random.randn(8, 4), index=index, columns=columns)

df3 = df.iloc[[0, 1, 4, 7], [1, 2]]

df3

exp                  B          
animal             dog       cat
first second                    
bar   one    -1.402028  2.456520
      two     0.167431  2.001867
foo   one     0.259946 -0.493744
qux   two     0.567028 -0.291046

In [25]:
df3.unstack()

exp            B                              
animal       dog                 cat          
second       one       two       one       two
first                                         
bar    -1.402028  0.167431  2.456520  2.001867
foo     0.259946       NaN -0.493744       NaN
qux          NaN  0.567028       NaN -0.291046

In [26]:
# The missing value can be filled with a specific value with the fill_value argument.
df3.unstack(fill_value = -1e9)

exp                B                                          
animal           dog                         cat              
second           one           two           one           two
first                                                         
bar    -1.402028e+00  1.674308e-01  2.456520e+00  2.001867e+00
foo     2.599460e-01 -1.000000e+09 -4.937437e-01 -1.000000e+09
qux    -1.000000e+09  5.670279e-01 -1.000000e+09 -2.910460e-01

### melt() and wide_to_long()
The top-level `melt()` function and the corresponding `DataFrame.melt()` are useful to reshape a `DataFrame` into a format where one or more columns are identifier variables, while all other columns, considered measured variables, are “unpivoted” to the row axis, leaving just two non-identifier columns, “variable” and “value”. The names of those columns can be customized by supplying the `var_name` and `value_name` parameters.

In [27]:
cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)

cheese

,first,last,height,weight
0,John,Doe,5.5,130
1,Mary,Bo,6.0,150


In [28]:
cheese.melt(id_vars=["first", "last"])

,first,last,variable,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


In [29]:
cheese.melt(id_vars=["first", "last"], var_name="quantity")

,first,last,quantity,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


When transforming a DataFrame using `melt()`, the index will be ignored. The original index values can be kept by setting the `ignore_index` parameter to `False` (default is `True`). `ignore_index=False` will however duplicate index values.

In [30]:
index = pd.MultiIndex.from_tuples([("person", "A"), ("person", "B")])

cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    },
    index=index,
)


cheese

first last  height  weight
person A  John  Doe     5.5     130
       B  Mary   Bo     6.0     150

In [31]:
cheese.melt(id_vars=["first", "last"])

,first,last,variable,value
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0


In [32]:
cheese.melt(id_vars=["first", "last"], ignore_index=False)

first last variable  value
person A  John  Doe   height    5.5
       B  Mary   Bo   height    6.0
       A  John  Doe   weight  130.0
       B  Mary   Bo   weight  150.0

In [33]:
# wide_to_long() is similar to melt() with more customization for column matching.
dft = pd.DataFrame(
    {
        "A1970": {0: "a", 1: "b", 2: "c"},
        "A1980": {0: "d", 1: "e", 2: "f"},
        "B1970": {0: 2.5, 1: 1.2, 2: 0.7},
        "B1980": {0: 3.2, 1: 1.3, 2: 0.1},
        "X": dict(zip(range(3), np.random.randn(3))),
    }
)

dft["id"] = dft.index
dft

,A1970,A1980,B1970,B1980,X,id
0,a,d,2.5,3.2,-0.282846,0
1,b,e,1.2,1.3,-0.220285,1
2,c,f,0.7,0.1,-0.750349,2


In [34]:
pd.wide_to_long(dft, ["A", "B"], i="id", j="year")

,,X,A,B
id,year,,,
0,1970,-0.282846,a,2.5
1,1970,-0.220285,b,1.2
2,1970,-0.750349,c,0.7
0,1980,-0.282846,d,3.2
1,1980,-0.220285,e,1.3
2,1980,-0.750349,f,0.1


### `crosstab()`

Use **`crosstab()`** to compute a cross-tabulation of two (or more) factors. By default **`crosstab()`** computes a frequency table of the factors unless an array of values and an aggregation function are passed.

Any **`Series`** passed will have their name attributes used unless row or column names for the cross-tabulation are specified.


In [35]:
a = np.array(["foo", "foo", "bar", "bar", "foo", "foo"], dtype=object)
b = np.array(["one", "one", "two", "one", "two", "one"], dtype=object)
c = np.array(["dull", "dull", "shiny", "dull", "dull", "shiny"], dtype=object)
pd.crosstab(a, [b, c], rownames=["a"], colnames=["b", "c"])

b    one        two      
c   dull shiny dull shiny
a                        
bar    1     0    0     1
foo    2     1    1     0

In [36]:
# If crosstab() receives only two Series, it will provide a frequency table.

df = pd.DataFrame(
    {'A': [1,2,2,2,2], 'B': [3,3,4,4,4], 'C':[1, 1, np.nan, 1, 1]}
)
df

,A,B,C
0,1,3,1.0
1,2,3,1.0
2,2,4,NaN
3,2,4,1.0
4,2,4,1.0


In [37]:
pd.crosstab(df['A'], df['B'])

B,3,4
A,,
1,1,0
2,1,3


In [38]:
# crosstab() can also summarize to Categorical data.

foo = pd.Categorical(['a', 'b'], categories = ['a', 'b', 'c'])
bar = pd.Categorical(['d', 'e'], categories = ['d', 'e', 'f'])
pd.crosstab(foo, bar)

col_0,d,e
row_0,,
a,1,0
b,0,1


In [39]:
# For Categorical data, to include all of data categories even if the actual data does not contain any instances of a particular category, use dropna=False.

pd.crosstab(foo, bar, dropna=False)

col_0,d,e,f
row_0,,,
a,1,0,0
b,0,1,0
c,0,0,0


### Normalization

Frequency tables can also be normalized to show percentages rather than counts using the normalize argument:

In [40]:
pd.crosstab(df["A"], df["B"], normalize=True)

B,3,4
A,,
1,0.2,0.0
2,0.2,0.6


In [41]:
# normalize can also normalize values within each row or within each column:
pd.crosstab(df["A"], df["B"], normalize="columns")

B,3,4
A,,
1,0.5,0.0
2,0.5,1.0


In [42]:
# crosstab() can also accept a third Series and an aggregation function (aggfunc) that will be applied to the values of the third Series within each group defined by the first two Series:
pd.crosstab(df["A"], df["B"], values=df["C"], aggfunc="sum")

B,3,4
A,,
1,1.0,NaN
2,1.0,2.0


### cut()
The cut() function computes groupings for the values of the input array and is often used to transform continuous variables to discrete or categorical variables:

An integer bins will form equal-width bins.

In [43]:
ages = np.array([10, 15, 13, 12, 23, 25, 28, 59, 60])

pd.cut(ages, bins=3)

[(9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (9.95, 26.667], (26.667, 43.333], (43.333, 60.0], (43.333, 60.0]]
Categories (3, interval[float64, right]): [(9.95, 26.667] < (26.667, 43.333] < (43.333, 60.0]]

In [44]:
# A list of ordered bin edges will assign an interval for each variable.
pd.cut(ages, bins=[0, 18, 35, 70])

[(0, 18], (0, 18], (0, 18], (0, 18], (18, 35], (18, 35], (18, 35], (35, 70], (35, 70]]
Categories (3, interval[int64, right]): [(0, 18] < (18, 35] < (35, 70]]

In [45]:
# If the bins keyword is an IntervalIndex, then these will be used to bin the passed data.
pd.cut(ages, bins=pd.IntervalIndex.from_breaks([0, 40, 70]))

[(0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (0, 40], (40, 70], (40, 70]]
Categories (2, interval[int64, right]): [(0, 40] < (40, 70]]

### get_dummies() and from_dummies()
To convert categorical variables of a Series into a “dummy” or “indicator”, get_dummies() creates a new DataFrame with columns of the unique variables and the values representing the presence of those variables per row.

In [46]:
df = pd.DataFrame({"key": list("bbacab"), "data1": range(6)})

pd.get_dummies(df["key"])

,a,b,c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [47]:
df["key"].str.get_dummies()

,a,b,c
0,0,1,0
1,0,1,0
2,1,0,0
3,0,0,1
4,1,0,0
5,0,1,0


In [48]:
# prefix adds a prefix to the column names which is useful for merging the result with the original DataFrame:
dummies = pd.get_dummies(df["key"], prefix="key")

dummies

,key_a,key_b,key_c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [49]:
df[["data1"]].join(dummies)

,data1,key_a,key_b,key_c
0,0,False,True,False
1,1,False,True,False
2,2,True,False,False
3,3,False,False,True
4,4,True,False,False
5,5,False,True,False


In [50]:
# This function is often used along with discretization functions like cut():
values = np.random.randn(10)

values

array([ 0.20166741,  0.63173614, -0.52620767, -0.46995898, -0.24406815,
        0.61057641,  0.08216927,  0.75870962, -0.39057616, -1.00465516])

In [51]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,True,False,False,False
1,False,False,False,True,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,True,False
6,True,False,False,False,False
7,False,False,False,True,False
8,False,False,False,False,False
9,False,False,False,False,False


In [52]:
# get_dummies() also accepts a DataFrame. By default, object, string, or categorical type columns are encoded as dummy variables with other columns unaltered.
df = pd.DataFrame({"A": ["a", "b", "a"], "B": ["c", "c", "b"], "C": [1, 2, 3]})

pd.get_dummies(df)
 

,C,A_a,A_b,B_b,B_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [53]:
# Specifying the columns keyword will encode a column of any type.
pd.get_dummies(df, columns=["A"])

,B,C,A_a,A_b
0,c,1,True,False
1,c,2,False,True
2,b,3,True,False


As with the **`Series`** version, you can pass values for the `prefix` and `prefix_sep`. By default the column name is used as the prefix and `_` as the prefix separator. You can specify `prefix` and `prefix_sep` in 3 ways:

* **string**: Use the same value for `prefix` or `prefix_sep` for each column to be encoded.
* **list**: Must be the same length as the number of columns being encoded.
* **dict**: Mapping column name to prefix.


In [54]:
simple = pd.get_dummies(df, prefix="new_prefix")

simple

,C,new_prefix_a,new_prefix_b,new_prefix_b,new_prefix_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [55]:
from_list = pd.get_dummies(df, prefix=["from_A", "from_B"])

from_list

,C,from_A_a,from_A_b,from_B_b,from_B_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [56]:
from_dict = pd.get_dummies(df, prefix={"B": "from_B", "A": "from_A"})

from_dict

,C,from_A_a,from_A_b,from_B_b,from_B_c
0,1,True,False,False,True
1,2,False,True,False,True
2,3,True,False,True,False


In [57]:
# To avoid collinearity when feeding the result to statistical models, specify drop_first=True.
s = pd.Series(list("abcaa"))

pd.get_dummies(s)

,a,b,c
0,True,False,False
1,False,True,False
2,False,False,True
3,True,False,False
4,True,False,False


In [58]:
pd.get_dummies(s, drop_first=True)

,b,c
0,False,False
1,True,False
2,False,True
3,False,False
4,False,False


In [59]:
# When a column contains only one level, it will be omitted in the result.
df = pd.DataFrame({"A": list("aaaaa"), "B": list("ababc")})
pd.get_dummies(df)

,A_a,B_a,B_b,B_c
0,True,True,False,False
1,True,False,True,False
2,True,True,False,False
3,True,False,True,False
4,True,False,False,True


In [60]:
pd.get_dummies(df, drop_first=True)

,B_b,B_c
0,False,False
1,True,False
2,False,False
3,True,False
4,False,True


In [61]:
# The values can be cast to a different type using the dtype argument.
df = pd.DataFrame({"A": list("abc"), "B": [1.1, 2.2, 3.3]})
pd.get_dummies(df, dtype=np.float32).dtypes

B      float64
A_a    float32
A_b    float32
A_c    float32
dtype: object

In [62]:
# from_dummies() converts the output of get_dummies() back into a Series of categorical values from indicator values.
df = pd.DataFrame({"prefix_a": [0, 1, 0], "prefix_b": [1, 0, 1]})

df

,prefix_a,prefix_b
0,0,1
1,1,0
2,0,1


In [63]:
pd.from_dummies(df, sep="_")

,prefix
0,b
1,a
2,b


In [64]:
# Dummy coded data only requires k - 1 categories to be included, in this case the last category is the default category. The default category can be modified with default_category.
df = pd.DataFrame({"prefix_a": [0, 1, 0]})

df

,prefix_a
0,0
1,1
2,0


In [65]:
pd.from_dummies(df, sep="_", default_category="b")

,prefix
0,b
1,a
2,b


### explode()
For a DataFrame column with nested, list-like values, explode() will transform each list-like value to a separate row. The resulting Index will be duplicated corresponding to the index label from the original row:

In [66]:
keys = ["panda1", "panda2", "panda3"]
values = [["eats", "shoots"], ["shoots", "leaves"], ["eats", "leaves"]]
df = pd.DataFrame({"keys": keys, "values": values})
df

,keys,values
0,panda1,"[eats, shoots]"
1,panda2,"[shoots, leaves]"
2,panda3,"[eats, leaves]"


In [67]:
df["values"].explode()

0      eats
0    shoots
1    shoots
1    leaves
2      eats
2    leaves
Name: values, dtype: str

In [68]:
# DataFrame.explode can also explode the column in the DataFrame.
df.explode("values")

,keys,values
0,panda1,eats
0,panda1,shoots
1,panda2,shoots
1,panda2,leaves
2,panda3,eats
2,panda3,leaves


In [69]:
# Series.explode() will replace empty lists with a missing value indicator and preserve scalar entries.
s = pd.Series([[1, 2, 3], "foo", [], ["a", "b"]])
s

0    [1, 2, 3]
1          foo
2           []
3       [a, b]
dtype: object

In [70]:
# A comma-separated string value can be split into individual values in a list and then exploded to a new row.
df = pd.DataFrame([{"var1": "a,b,c", "var2": 1}, {"var1": "d,e,f", "var2": 2}])

df.assign(var1=df.var1.str.split(",")).explode("var1")

,var1,var2
0,a,1
0,b,1
0,c,1
1,d,2
1,e,2
1,f,2
